#### station_master.csv 보강 (이름 + 위도/경도 추가)

station_demand_model.pkl은 이미 학습이 끝난 상태 그대로 두고, station_master.csv만 이름/좌표를 채워서 다시 저장


In [1]:
import pandas as pd
import numpy as np


#### 실제 발급받은 좌표 데이터 사용

발급받은 파일은 `대여소_ID`(예: 'ST-999'), `주소1`, `주소2`, `위도`, `경도` 컬럼으로 구성되어 있고,
`station_name`은 들어있지 않습니다(주소만 있음). 그리고 ID가 `ST-999`처럼 문자열이라 기존
`station_master.csv`의 숫자 ID(예: 102)와 형식이 달라서 변환이 필요합니다.

좌표가 (0, 0)으로 비어있는 행(77개)은 결측치로 처리해서 제외합니다.

In [3]:
# TODO: 실제 업로드한 파일 경로로 수정
coord_source = pd.read_csv(
    '../data/서울시 공공자전거 따릉이 대여소 마스터 정보.csv',
    encoding='cp949'
)
coord_source.head()


,대여소_ID,주소1,주소2,위도,경도
0,ST-999,서울특별시 양천구 목동서로 280,목동아파트 8단지 상가동,0.000000,0.000000
1,ST-998,서울특별시 양천구 목동서로 130,목동아파트 4단지 상가동,0.000000,0.000000
2,ST-997,서울특별시 양천구 목동중앙로 49,목동3단지 시내버스정류장,37.534390,126.869598
3,ST-996,서울특별시 양천구 남부순환로88길5-16,양강중학교앞 교차로,37.524334,126.850548
4,ST-995,서울특별시 양천구 중앙로 153 공중화장실,NaN,37.510597,126.857323


In [4]:
# 'ST-999' -> 999 로 ID 형식을 기존 station_master.csv와 맞춤
coord_source['station_id'] = (
    coord_source['대여소_ID'].str.replace('ST-', '', regex=False).astype(int)
)

coord_source = coord_source.rename(columns={'위도': 'latitude', '경도': 'longitude'})

# 좌표가 (0, 0)인 행은 결측치로 처리
invalid = (coord_source['latitude'] == 0) & (coord_source['longitude'] == 0)
print(f'좌표 결측 처리된 행: {invalid.sum()}개')
coord_source.loc[invalid, ['latitude', 'longitude']] = np.nan

coord_lookup = coord_source[['station_id', 'latitude', 'longitude']].drop_duplicates(subset='station_id')
coord_lookup.head()


좌표 결측 처리된 행: 77개


,station_id,latitude,longitude
0,999,NaN,NaN
1,998,NaN,NaN
2,997,37.534390,126.869598
3,996,37.524334,126.850548
4,995,37.510597,126.857323


#### 최종: station_master.csv 다시 저장 (이름 + 좌표 포함)
coord_lookup 그대로 사용

In [5]:
# 이름을 위해 원본 station 데이터도 다시 불러옴 (02_station_demand_model.ipynb와 동일)
station_raw = pd.read_excel('../data/공공자전거 대여소 정보(26.6월 기준).xlsx', header=3)
station_raw.columns = ['station_id','station_name','district','addr1','addr2','addr3',
                        'install_date','install_type','rack_count','operation_type']
station_raw = station_raw.dropna(subset=['station_id'])
station_raw['station_id'] = station_raw['station_id'].astype(int)
station_raw = station_raw[['station_id','station_name','district','rack_count']].drop_duplicates(subset='station_id')

station_master = station_raw.merge(coord_lookup, on='station_id', how='left')

missing = station_master['latitude'].isna().sum()
print(f'좌표를 못 찾은 대여소 수: {missing} / {len(station_master)}')
station_master.head()


좌표를 못 찾은 대여소 수: 1025 / 2789


,station_id,station_name,district,rack_count,latitude,longitude
0,102,망원역 1번출구 앞,마포구,15.0,37.537010,127.082245
1,103,망원역 2번출구 앞,마포구,14.0,37.549061,127.057793
2,104,합정역 1번출구 앞,마포구,13.0,37.548203,127.057114
3,105,합정역 5번출구 앞,마포구,5.0,37.545170,127.057571
4,106,합정역 7번출구 앞,마포구,12.0,37.539654,127.052589


In [6]:
# station_demand_model.pkl, district_encoder.pkl은 건드리지 않고 station_master.csv만 덮어씀
station_master.to_csv('../models/station_master.csv', index=False)
print('저장 완료: station_master.csv (station_name, latitude, longitude 포함)')


저장 완료: station_master.csv (station_name, latitude, longitude 포함)
